<a href="https://colab.research.google.com/github/prasannakumar9i/ev-llm-diagnostic-assistant/blob/main/ev_rag_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EV LLM Diagnostic Assistant (RAG Project)

Author: Sunny  
Project: EV AI Diagnostic Assistant  
Tools: Python, LangChain, FAISS, HuggingFace

## Day 1 – Project Setup

Goal:
- Create GitHub repository
- Connect Google Colab
- Initialize project notebook

#Day 2 Install libraries

In [ ]:
!pip install langchain
!pip  install sentence-transformers
!pip install faiss-cpu
!pip install pypdf
!pip install transformers

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded successfully")

In [ ]:
sentence = "EV battery draining quickly"

embedding = model.encode(sentence)

print(len(embedding))

# Day 3 — Download EV Repair Manuals Dataset

In this step, we collect EV repair manuals in PDF format.
These manuals will act as the knowledge base for the AI diagnostic assistant.
Later we will extract text from these PDFs and convert them into embeddings.

In [22]:
import os

os.makedirs("ev_manuals", exist_ok=True)

print("EV manuals dataset folder created successfully")

EV manuals dataset folder created successfully


In [23]:
import os
os.listdir()

['.config', 'ev_manuals', 'sample_data']

# Day 4 — Load PDF Manuals and Extract Text

In this step we upload a PDF manual into Google Colab and use
LangChain's PyPDFLoader to read the document.

Each page of the PDF will be converted into a document object
that will later be split into chunks for the RAG system.

#install pdf Libraries

In [25]:
from google.colab import files
uploaded = files.upload()

Saving harrier-ev-onwers-manual.pdf to harrier-ev-onwers-manual.pdf


In [26]:
!pip install langchain
!pip install langchain-community
!pip install pypdf

In [27]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("harrier-ev-onwers-manual.pdf")

documents = loader.load()

print("Total pages loaded:", len(documents))

Total pages loaded: 422


In [31]:
print(documents[0].page_content[:500])

Following items are provided 
with your vehicle: 
1. Owner’s Manual
2. Battery Warranty Card
(if applicable)
3. First Aid Kit
4. Advance Warning Triangle
5. Jack
6. Spare Fuses (Provided in
fuse box)
7. Tool Kit
CAR IDENTIFICATION RECORD 
OWNER’S NAME: _____________________________________________________ 
ADDRESS: __________________________________________________________ 
SELLING DEALER CODE: _______________________________________________ 
DATE OF DELIVERY: ___________________________________


# Day 5 — Split Documents into Text Chunks

In this step we split the extracted manual text into smaller chunks.
Chunking improves retrieval accuracy in RAG systems because the
language model searches smaller text segments instead of full documents.

In [35]:
!pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [37]:
# creating text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20
)

#split the documents

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))


Total chunks: 804


In [42]:
print(chunks[0].page_content[:1000])


Following items are provided 
with your vehicle: 
1. Owner’s Manual
2. Battery Warranty Card
(if applicable)
3. First Aid Kit
4. Advance Warning Triangle
5. Jack
6. Spare Fuses (Provided in
fuse box)
7. Tool Kit
CAR IDENTIFICATION RECORD 
OWNER’S NAME: _____________________________________________________ 
ADDRESS: __________________________________________________________ 
SELLING DEALER CODE: _______________________________________________ 
DATE OF DELIVERY: ___________________________________________________ 
DATE OF REGISTRATION: _______________________________________________ 
REGISTRATION NO.: ___________________________________________________ 
CHASSIS NO.: ________________________________________________________ 
MOTOR NO.: _________________________________________________________ 
       : _________________________________________________________ 
TRANSAXLE NO.: ______________________________________________________


#Day 6 Generate Embeddings

In [57]:
#packages
!pip install langchain
!pip install sentence-transformers
!pip install langchain-community
!pip install langchain-text-splitters

In [58]:
# imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [59]:
#loading the pdf
loader = PyPDFLoader("harrier-ev-onwers-manual.pdf")
documents = loader.load()

In [60]:
# splitting the text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print(len(docs))

1488


In [61]:
#creating embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [62]:
#generating vectors
texts = [doc.page_content for doc in docs]

vectors = embeddings.embed_documents(texts)

print(len(vectors))
print(len(vectors[0]))

1488
384


#Day 7 creating vector database

In [63]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 27.0 MB/s eta 0:00:00


In [64]:
#importing faiss
from langchain_community.vectorstores import FAISS

In [65]:
#create vector database
vectorstore = FAISS.from_documents(docs, embeddings)

In [68]:
#testing
query = "battery not charging"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("------")


 Switch off the AC supply. 
 Remove the charging gun from the charging inlet. 
 Wait for 5 minutes. 
 Restart the charging. (Refer charging procedure).
------
charger is connected. Contact the TATA MOTORS EV Authorised Service 
Centre to get the charging fail issue resolved. 
Charger Con-
nected Blue 
 
This symbol lights up as soon as the charger is  connected for charging the 
battery.
------
5 Check all the HV cables for loose-
ness, cuts, wear & tear To be done at every service 
6 
Clean Charging socket area.  
Also clean dust around hinge area of 
Charging Port Housing lid. 
To be done at every service 
10 Replace HV Battery breather plug.        R     R      R      R      R
------


In [69]:
vectorstore.save_local("ev_manual_vector_db")

In [70]:
query = "battery overheating"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("------")

ing fan. 
HV Battery Life & Maintenance 
This Vehicle comes with a standard bat-
tery warranty as mentioned in warranty 
section. Regular service of the vehicle 
and charging protocol to be followed to 
maximize the battery life. 
Energy Information 
The vehicle battery pack has a maximum 
energy as specified in Tec hnical Specifi-
cation. Energy retention capacity deterio-
rates over several cycles of usage and 
hence range deterioration happens over 
the time.
------
smoking materials away from the 
battery. 
 The battery contains sulphuric 
acid (electrolyte) which is poison-
ous and highly corrosive in nature. 
Getting electrolyte in your eyes or 
on the skin can cause severe 
burns. Wear protective clothing 
and a face shield or have a skilled 
technician to do the battery 
maintenance. 
TYRES 
 
1 Under inflation 
Excessive 
side tread 
wear 
2 Correct tyre pres-
sure 
Uniform 
wear 
3 Over inflation 
Excessive 
center 
tread wear
------
tow the vehicle without guidance from 
th

In [71]:
query = "charging port problem"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("------")

 Switch off the AC supply. 
 Remove the charging gun from the charging inlet. 
 Wait for 5 minutes. 
 Restart the charging. (Refer charging procedure).
------
ing the handle), the internal wires may 
be disconnected or get damaged. 
This may lead to electric shock or fire. 
 Do not leave th e vehicle with the 
charging flap open. An open charging 
flap may indicate that the vehicle door 
has been unlocked and may be sub-
ject to vehicle theft.  
 Always keep the charging connector 
and charging plug in clean and dry 
condition. Be sure to keep the charg-
ing cable in a condition where there is  
no water or moisture.
------
CHARGING 
58 
 
4. Open the charger inlet flap.  
5. Before connecting the charging gun 
to vehicle charging socket, make 
sure the gun lock is released.  
CAUTION 
If the Gun Lock is not released please 
don’t insert the Charging Gun force-
fully into the socket. It may damage 
the Charging Socket. 
6. Remove any dust on the Charging 
Gun and Charging Inlet. 

In [72]:
from google.colab import files
files.download("ev_manual_vector_db/index.faiss")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>